# nb04_train : 학습 실험 (E0 스모크 → E1 기준선 → E2 2단계 학습 → E3·E4 입력 비교)

- **가속기 : GPU (P100 또는 T4 x1)** · **인터넷 : ON** (첫 실행 시 사전학습 가중치 다운로드)
- **입력** : Dataset `hms-processed`, `hms-src`
- **출력** : `/kaggle/working/models/<EXP>/` (fold별 가중치, OOF 예측, 지표) → 실행 후 Dataset `hms-models`

| 실험 | 입력 | 모델 | 가설 | 실행 설정 |
|---|---|---|---|---|
| **E0** | 10분 스펙트로그램 | EfficientNet-B0 | 파이프라인 점검 | `EXP="E0"` : fold 0, 1 epoch, 512개 |
| **E1** | 10분 스펙트로그램 | EfficientNet-B0 | 기준선 | `EXP="E1"` : 5 fold, 5 epoch, 전체 |
| **E2** | 10분 스펙트로그램 | EfficientNet-B0 | **H1** : 고득표 미세조정 | `EXP="E2"` : 1단계 전체 5 epoch → 2단계 고득표(≥10표) 3 epoch, 학습률 1/10 |
| **E3** | **50초 EEG 스펙트로그램** (GPU에서 STFT, 400x256) | EfficientNet-B0 | **H2** : 입력 비교 | `EXP="E3"` : E2와 같은 2단계 학습 |
| **E4** | **10분 + 50초 가로 연결** (400x556) | EfficientNet-B0 | **H2** : 결합 입력 우수 | `EXP="E4"` : E2와 같은 2단계 학습 |

- **E2 판정 방식** : 같은 실행 안에서 1단계(E1과 같은 조건)와 2단계 OOF를 모두 기록해 비교 → 실행 간 GPU 변동을 배제한 통제 비교
- **선택 기준** : E2부터 코드북 5장대로 **고득표 검증 KL**이 가장 낮은 epoch 저장 (E0·E1은 실행 당시 기준인 전체 KL 유지)

- **H2 판정** : E4 고득표 KL이 E2(10분 단독)와 E3(50초 단독) 중 좋은 쪽보다 0.01 이상 낮으면 지지 (세 실험 모두 같은 학습 방식·fold)
- **50초 스펙트로그램** : `src/models/eeg_spec.py` — 16채널 STFT(2.56초 창, 0.2초 간격) → 체인별 전극쌍 평균 → 0.5~20Hz → 로그 → 체인 4개 세로 연결 (대회 스펙트로그램과 같은 배치)

**실행 순서** : 새 실험은 `SMOKE = True`로 대화형 점검(약 3분, fold 0·1 epoch·512개) → 오류 없으면 `SMOKE = False`로 바꿔 Save Version

## 0. 설정 (목적 : 실험 조건 고정 / 출력 : CFG, 경로)

In [1]:
EXP = "E4"            # "E0" : 스모크 / "E1" : 기준선 / "E2" : 2단계 학습 (H1) / "E3" : 50초 EEG 입력 / "E4" : 결합 입력 (H2)
SMOKE = False          # True : 대화형 점검 (fold 0, 각 단계 1 epoch, 512개) → 확인 후 False로 바꿔 Save Version

import sys, os, json, time, math, copy
from pathlib import Path

def find_root(marker, root="/kaggle/input"):
    for pattern in [f"*/{marker}", f"*/*/{marker}", f"*/*/*/{marker}", f"*/*/*/*/{marker}"]:
        hits = sorted(Path(root).glob(pattern))
        if hits:
            return hits[0].parent
    raise FileNotFoundError(f"{marker} 를 찾지 못했습니다. Add Input 연결을 확인하세요.")

SRC_ROOT = Path(os.environ.get("HMS_SRC_DIR") or find_root("src/data/targets.py").parents[1])
PROC_DIR = Path(os.environ.get("HMS_PROC_DIR") or find_root("eeg50.npy"))
sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from src.config import load_config
from src.data.targets import PROB_COLS, CLASSES
from src.data.folds import check_no_leak
from src.eval.metrics import evaluate
from src.models.eeg_spec import HMSNet

CFG = load_config(SRC_ROOT / "configs" / "base.yaml")
SETTINGS = {
    "E0": dict(folds=[0], epochs=1, n_limit=512, select="kl_all", stage2=None),
    "E1": dict(folds=[0, 1, 2, 3, 4], epochs=5, n_limit=None, select="kl_all", stage2=None),
    "E2": dict(folds=[0, 1, 2, 3, 4], epochs=5, n_limit=None, select="kl_high",
               stage2=dict(epochs=3, lr_mult=0.1)),          # 2단계 : 고득표 구간만, 학습률 1/10
}
SETTINGS["E3"] = dict(SETTINGS["E2"], input="eeg")           # E2와 같은 학습, 입력만 50초 EEG 스펙트로그램
SETTINGS["E4"] = dict(SETTINGS["E2"], input="both")          # E2와 같은 학습, 10분 + 50초 가로 연결
SETTINGS = dict(SETTINGS[EXP])
SETTINGS.setdefault("input", "spec")
if SMOKE:                                                    # 같은 코드 경로를 작게 점검 (결과는 *_smoke 폴더, 기록 대상 아님)
    SETTINGS.update(folds=[0], epochs=1, n_limit=512)
    if SETTINGS["stage2"]:
        SETTINGS["stage2"] = dict(SETTINGS["stage2"], epochs=1)
HYPOTHESIS = {"E0": "smoke", "E1": "baseline", "E2": "H1 high-vote fine-tuning",
              "E3": "H2 50s EEG spectrogram input", "E4": "H2 combined input (10min + 50s)"}[EXP]
INPUT_DESC = {"spec": "Kaggle spectrogram 4x100x300",
              "eeg": "50s EEG STFT spectrogram 4x100x256 (GPU)",
              "both": "Kaggle 10min 400x300 + 50s EEG 400x256 concat (400x556)"}[SETTINGS["input"]]
HP = dict(model="efficientnet_b0", pretrained=True, batch_size=32, lr=1e-3, weight_decay=1e-2,
          num_workers=2, seed=CFG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = Path(os.environ.get("HMS_OUT_DIR", "/kaggle/working")) / "models" / (EXP + ("_smoke" if SMOKE and EXP != "E0" else ""))
OUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(HP["seed"]); np.random.seed(HP["seed"])
print(f"EXP={EXP} | SMOKE={SMOKE} | 입력={SETTINGS['input']} | DEVICE={DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")
print("SRC_ROOT :", SRC_ROOT, "\nPROC_DIR :", PROC_DIR, "\nOUT_DIR  :", OUT_DIR)
if DEVICE == "cpu":
    print("[경고] GPU가 꺼져 있습니다. 오른쪽 패널 Accelerator를 GPU로 바꾸세요.")

EXP=E4 | SMOKE=False | 입력=both | DEVICE=cuda (Tesla T4)
SRC_ROOT : /kaggle/input/datasets/sungik4057/hms-src 
PROC_DIR : /kaggle/input/datasets/sungik4057/hms-processed/hms_processed 
OUT_DIR  : /kaggle/working/models/E4


## 1. 데이터 (목적 : hms-processed 로드·검증 / 출력 : meta, SPEC·EEG 배열)

In [2]:
USE_SPEC, USE_EEG = SETTINGS["input"] in ("spec", "both"), SETTINGS["input"] in ("eeg", "both")
meta = pd.read_csv(PROC_DIR / "meta.csv")
spec_all = np.load(PROC_DIR / "spec10.npy", mmap_mode="r")
eeg_all = np.load(PROC_DIR / "eeg50.npy", mmap_mode="r")
assert len(meta) == spec_all.shape[0] == eeg_all.shape[0], "meta.csv와 배열 행 수가 다릅니다"
assert np.allclose(meta[PROB_COLS].sum(axis=1), 1)
check_no_leak(meta)

if SETTINGS["n_limit"]:
    meta = meta.sample(SETTINGS["n_limit"], random_state=HP["seed"])
meta = meta.reset_index().rename(columns={"index": "row"})   # row : 배열에서의 위치

# 배열을 메모리에 한 번에 올림 : Dataset 디스크에서 무작위로 읽으면 매우 느림 (E0 1 epoch 155초의 원인)
t0 = time.time()
def load_rows(name, mm):
    if mm is None:
        return None
    if SETTINGS["n_limit"]:
        return np.asarray(mm[np.sort(meta["row"].to_numpy())])   # 필요한 행만 순서대로 읽기
    return np.load(PROC_DIR / name)                              # 전체 순차 읽기 (spec 약 4GB, eeg 약 5.5GB)
SPEC = load_rows("spec10.npy", spec_all if USE_SPEC else None)
EEG = load_rows("eeg50.npy", eeg_all if USE_EEG else None)
if SETTINGS["n_limit"]:
    rows = np.sort(meta["row"].to_numpy())
    meta["row"] = np.searchsorted(rows, meta["row"].to_numpy())  # 메모리 배열 기준 위치로 변경
for name, a in [("SPEC", SPEC), ("EEG", EEG)]:
    if a is not None:
        print(f"{name} 메모리 적재 {a.shape} {a.dtype} {a.nbytes / 1e9:.1f}GB")
print(f"학습 대상 {len(meta):,}개 | 입력 {SETTINGS['input']} | 적재 {time.time() - t0:.0f}s")
display(meta.groupby("fold").agg(n=("eeg_id", "size"), high_vote=("high_vote", "mean")).round(3))

SPEC 메모리 적재 (17089, 4, 100, 300) float16 4.1GB
EEG 메모리 적재 (17089, 16, 10000) float16 5.5GB
학습 대상 17,089개 | 입력 both | 적재 113s


,n,high_vote
fold,,
0,3345,0.361
1,3389,0.360
2,3461,0.335
3,3277,0.382
4,3617,0.304


In [3]:
EMPTY = torch.zeros(0)

class HMSDataset(torch.utils.data.Dataset):
    """(spec, eeg, y). spec : 10분 스펙트로그램 4부위를 세로로 이은 표준화 이미지 (1, 400, 300)
    eeg : 50초 몽타주 (16, 10000) → 모델 안에서 GPU STFT. 쓰지 않는 입력은 빈 텐서."""
    def __init__(self, df, spec_arr, eeg_arr):
        self.rows = df["row"].to_numpy()
        self.y = np.array(df[PROB_COLS].to_numpy(dtype=np.float32), copy=True)   # 쓰기 가능한 복사본 (pandas 읽기 전용 경고 방지)
        self.spec, self.eeg = spec_arr, eeg_arr  # 메모리 배열 (워커는 fork로 복사 없이 공유)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        s = e = EMPTY
        if self.spec is not None:
            x = np.asarray(self.spec[r], dtype=np.float32).reshape(-1, 300)   # (400, 300)
            s = torch.from_numpy(((x - x.mean()) / (x.std() + 1e-6))[None])   # 표본별 표준화 : 기록 간 전력 수준 차이 보정
        if self.eeg is not None:
            e = torch.from_numpy(np.asarray(self.eeg[r], dtype=np.float32))
        return s, e, torch.from_numpy(self.y[i])

s0, e0, y0 = HMSDataset(meta.head(4), SPEC, EEG)[0]
print("spec", tuple(s0.shape), "| eeg", tuple(e0.shape), "| 타깃 합", float(y0.sum()))

spec (1, 400, 300) | eeg (16, 10000) | 타깃 합 1.0


## 2. 모델·학습 함수 (목적 : 동일 조건 학습 루프 / 출력 : train_fold 함수)

In [4]:
def make_model(pretrained):
    try:
        bb = timm.create_model(HP["model"], pretrained=pretrained, in_chans=1, num_classes=len(CLASSES))
    except Exception as e:
        print(f"[경고] 사전학습 가중치 로드 실패 → 무작위 초기화로 진행 ({type(e).__name__}). 인터넷 설정을 확인하세요.")
        bb = timm.create_model(HP["model"], pretrained=False, in_chans=1, num_classes=len(CLASSES))
    return HMSNet(bb, SETTINGS["input"])            # 저장은 backbone만 (E1·E2 가중치 형식과 동일)

def to_dev(s, e):
    return (s.to(DEVICE, non_blocking=True) if s.numel() else None,
            e.to(DEVICE, non_blocking=True) if e.numel() else None)

def kl_loss(logits, target):
    """대회 지표와 같은 방향의 손실 : KL(target || softmax(logits))."""
    return F.kl_div(F.log_softmax(logits, dim=1), target, reduction="batchmean")

@torch.no_grad()
def predict(model, loader):
    model.eval(); out = []
    for s, e, _ in loader:
        with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
            out.append(F.softmax(model(*to_dev(s, e)).float(), dim=1).cpu().numpy())
    return np.concatenate(out)

def make_loader(d, shuffle):
    return torch.utils.data.DataLoader(HMSDataset(d, SPEC, EEG), batch_size=HP["batch_size"], shuffle=shuffle,
                                       num_workers=HP["num_workers"], pin_memory=DEVICE == "cuda", drop_last=shuffle)

def run_stage(model, tr, va, epochs, lr, tag, k):
    """한 단계 학습. 검증 fold 전체로 매 epoch 평가하고 SETTINGS['select'] 기준 최적 epoch의 예측·가중치를 반환."""
    dl_tr, dl_va = make_loader(tr, True), make_loader(va, False)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=HP["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=max(epochs * len(dl_tr), 20), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    best, best_pred, best_state = float("inf"), None, None
    for ep in range(epochs):
        model.train(); t0, losses = time.time(), []
        for s, e, y in dl_tr:
            y = y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
                loss = kl_loss(model(*to_dev(s, e)).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            losses.append(loss.item())
        pred = predict(model, dl_va)
        m = evaluate(va, pred)
        print(f"  fold {k}{(' ' + tag) if tag else ''} ep {ep + 1}/{epochs} | n_train {len(tr):,} | train {np.mean(losses):.4f} | "
              f"val KL {m['kl_all']:.4f} (고득표 {m['kl_high']:.4f}) | {time.time() - t0:.0f}s", flush=True)
        if m[SETTINGS["select"]] < best:           # 선택 기준 최소 epoch 보관
            best, best_pred = m[SETTINGS["select"]], pred
            best_state = copy.deepcopy({kk: v.detach().cpu() for kk, v in model.backbone.state_dict().items()})
    return best_pred, best_state

def train_fold(k):
    tr, va = meta[meta.fold != k], meta[meta.fold == k]
    model = make_model(HP["pretrained"]).to(DEVICE)
    pred1, state1 = run_stage(model, tr, va, SETTINGS["epochs"], HP["lr"], "1단계" if SETTINGS["stage2"] else "", k)
    if not SETTINGS["stage2"]:
        torch.save(state1, OUT_DIR / f"fold{k}.pt")
        return va.index, pred1, None
    torch.save(state1, OUT_DIR / f"fold{k}_stage1.pt")
    s2 = SETTINGS["stage2"]
    model.backbone.load_state_dict(state1)         # 1단계 최적 가중치에서 시작
    tr_hv = tr[tr.high_vote == 1]                  # 고득표(≥10표) 구간만으로 미세조정
    pred2, state2 = run_stage(model, tr_hv, va, s2["epochs"], HP["lr"] * s2["lr_mult"], "2단계", k)
    torch.save(state2, OUT_DIR / f"fold{k}.pt")    # 제출용 최종 가중치
    return va.index, pred1, pred2

## 3. 학습 실행 (목적 : fold별 학습·OOF 수집 / 출력 : fold 가중치, oof.csv)

In [5]:
oof = np.full((len(meta), len(CLASSES)), np.nan, dtype=np.float32)
oof_s1 = oof.copy()                                   # E2 : 1단계 OOF (H1 비교용)
t_start = time.time()
for k in SETTINGS["folds"]:
    t0 = time.time()
    idx, pred1, pred2 = train_fold(k)
    oof[idx] = pred2 if pred2 is not None else pred1
    oof_s1[idx] = pred1
    print(f"fold {k} 완료 {(time.time() - t0) / 60:.1f}분 | 누적 {(time.time() - t_start) / 60:.1f}분", flush=True)
gpu_hours = (time.time() - t_start) / 3600
print(f"전체 학습 {gpu_hours * 60:.1f}분 (GPU {gpu_hours:.2f}시간)")

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_23/2273105322.py:43: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  fold 0 1단계 ep 1/5 | n_train 13,744 | train 0.8536 | val KL 0.6873 (고득표 0.4580) | 193s
  fold 0 1단계 ep 2/5 | n_train 13,744 | train 0.4862 | val KL 0.6750 (고득표 0.4146) | 179s
  fold 0 1단계 ep 3/5 | n_train 13,744 | train 0.3418 | val KL 0.7149 (고득표 0.4364) | 179s
  fold 0 1단계 ep 4/5 | n_train 13,744 | train 0.1970 | val KL 0.6859 (고득표 0.4972) | 179s
  fold 0 1단계 ep 5/5 | n_train 13,744 | train 0.1141 | val KL 0.6934 (고득표 0.4933) | 179s
  fold 0 2단계 ep 1/3 | n_train 4,732 | train 0.2325 | val KL 0.6539 (고득표 0.3111) | 70s
  fold 0 2단계 ep 2/3 | n_train 4,732 | train 0.1733 | val KL 0.6441 (고득표 0.3079) | 69s
  fold 0 2단계 ep 3/3 | n_train 4,732 | train 0.1444 | val KL 0.6441 (고득표 0.3198) | 69s
fold 0 완료 18.7분 | 누적 18.7분


/tmp/ipykernel_23/2273105322.py:43: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


...

  fold 2 2단계 ep 1/3 | n_train 4,780 | train 0.2369 | val KL 0.5714 (고득표 0.3169) | 71s
  fold 2 2단계 ep 2/3 | n_train 4,780 | train 0.1710 | val KL 0.5705 (고득표 0.3112) | 70s
  fold 2 2단계 ep 3/3 | n_train 4,780 | train 0.1416 | val KL 0.5738 (고득표 0.3082) | 70s
fold 2 완료 18.4분 | 누적 55.6분


/tmp/ipykernel_23/2273105322.py:43: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  fold 3 1단계 ep 1/5 | n_train 13,812 | train 0.8772 | val KL 0.7215 (고득표 0.5248) | 185s
  fold 3 1단계 ep 2/5 | n_train 13,812 | train 0.4816 | val KL 0.6332 (고득표 0.4312) | 180s
  fold 3 1단계 ep 3/5 | n_train 13,812 | train 0.3478 | val KL 0.6726 (고득표 0.4544) | 180s
  fold 3 1단계 ep 4/5 | n_train 13,812 | train 0.1959 | val KL 0.6762 (고득표 0.4845) | 180s
  fold 3 1단계 ep 5/5 | n_train 13,812 | train 0.1147 | val KL 0.6724 (고득표 0.4886) | 180s
  fold 3 2단계 ep 1/3 | n_train 4,687 | train 0.2406 | val KL 0.6542 (고득표 0.3220) | 69s
  fold 3 2단계 ep 2/3 | n_train 4,687 | train 0.1748 | val KL 0.6106 (고득표 0.3232) | 69s
  fold 3 2단계 ep 3/3 | n_train 4,687 | train 0.1456 | val KL 0.6173 (고득표 0.3235) | 69s
fold 3 완료 18.5분 | 누적 74.1분


/tmp/ipykernel_23/2273105322.py:43: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  fold 4 1단계 ep 1/5 | n_train 13,472 | train 0.8571 | val KL 0.7138 (고득표 0.4813) | 184s
  fold 4 1단계 ep 2/5 | n_train 13,472 | train 0.4823 | val KL 0.6977 (고득표 0.4289) | 178s
  fold 4 1단계 ep 3/5 | n_train 13,472 | train 0.3430 | val KL 0.6577 (고득표 0.4560) | 177s
  fold 4 1단계 ep 4/5 | n_train 13,472 | train 0.1897 | val KL 0.6697 (고득표 0.4647) | 176s
  fold 4 1단계 ep 5/5 | n_train 13,472 | train 0.1096 | val KL 0.6719 (고득표 0.4718) | 178s
  fold 4 2단계 ep 1/3 | n_train 4,838 | train 0.2311 | val KL 0.6571 (고득표 0.3397) | 72s
  fold 4 2단계 ep 2/3 | n_train 4,838 | train 0.1674 | val KL 0.6350 (고득표 0.3386) | 72s
  fold 4 2단계 ep 3/3 | n_train 4,838 | train 0.1373 | val KL 0.6394 (고득표 0.3353) | 72s
fold 4 완료 18.5분 | 누적 92.6분
전체 학습 92.6분 (GPU 1.54시간)


## 4. 평가·기록 (목적 : 코드북 지표 산출 / 출력 : metrics.json, results_row.csv)

In [6]:
done = ~np.isnan(oof).any(axis=1)
ev = meta[done]
metrics = evaluate(ev, oof[done], target_spec=CFG["eval"]["target_specificity"])
per_fold = {int(k): round(evaluate(ev[ev.fold == k], oof[done][(ev.fold == k).to_numpy()])["kl_all"], 4)
            for k in SETTINGS["folds"]}

def save_oof(arr, name):
    pd.concat([ev[["eeg_id", "patient_id", "fold", "high_vote"] + PROB_COLS].reset_index(drop=True),
               pd.DataFrame(arr[done], columns=[f"pred_{c}" for c in CLASSES])], axis=1).to_csv(OUT_DIR / name, index=False)
save_oof(oof, "oof.csv")

verdict = None
if SETTINGS["stage2"]:                                 # H1 : 같은 실행 안의 1단계 대비 2단계 개선량
    save_oof(oof_s1, "oof_stage1.csv")
    m1 = evaluate(ev, oof_s1[done], target_spec=CFG["eval"]["target_specificity"])
    delta = m1["kl_high"] - metrics["kl_high"]
    verdict = "지지" if delta >= 0.03 else ("방향 일치·약함" if delta > 0 else "기각")
    cmp = pd.DataFrame({"1단계(전체)": m1, "2단계(고득표 미세조정)": metrics}).loc[
        ["kl_all", "kl_high", "auc_lat", "auc_pd", "sens_at_spec90"]].astype(float).round(4)
    display(cmp)
    if EXP == "E2":
        print(f"H1 판정 : 고득표 KL 개선 {delta:+.4f} (기준 0.03 이상) → {verdict}")
    else:                                              # E3·E4 : 2단계 효과는 참고 (H1 재현 여부), 판정 대상은 H2
        print(f"2단계 효과(H1 재현) : 고득표 KL 개선 {delta:+.4f} → {verdict} | H2는 E2·E3·E4 고득표 KL 비교로 판정")
        verdict = f"stage2 gain {delta:+.4f}; H2 pending (compare E2/E3/E4)"

record = {"exp": EXP, "hypothesis": HYPOTHESIS,
          "change": f"{INPUT_DESC} -> EfficientNet-B0 (in_chans=1)"
                    + (" | stage2: high-vote fine-tune 3ep, lr x0.1" if SETTINGS["stage2"] else ""),
          "cv_kl_all": round(metrics["kl_all"], 4), "cv_kl_high": round(metrics["kl_high"], 4),
          "public_kl": None, "private_kl": None,
          "auc_lat": round(metrics["auc_lat"], 4), "auc_pd": round(metrics["auc_pd"], 4),
          "sens_at_spec90": round(metrics["sens_at_spec90"], 4), "gpu_hours": round(gpu_hours, 2),
          "kaggle_version": None, "verdict": verdict,
          "note": f"per-fold KL {per_fold}" + (f" | stage1 high KL {m1['kl_high']:.4f}" if SETTINGS["stage2"] else "")}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump({"record": record, "hp": HP, "settings": SETTINGS, "smoke": SMOKE, "full": metrics}, f, indent=2, default=float)
pd.DataFrame([record]).to_csv(OUT_DIR / "results_row.csv", index=False)

display(pd.DataFrame([record]).T.rename(columns={0: EXP}))
print("저장 파일 :", sorted(p.name for p in OUT_DIR.iterdir()))

,1단계(전체),2단계(고득표 미세조정)
kl_all,0.6387,0.6173
kl_high,0.4198,0.3228
auc_lat,0.9229,0.9355
auc_pd,0.9148,0.9234
sens_at_spec90,0.8798,0.9454


2단계 효과(H1 재현) : 고득표 KL 개선 +0.0970 → 지지 | H2는 E2·E3·E4 고득표 KL 비교로 판정


,E4
exp,E4
hypothesis,H2 combined input (10min + 50s)
change,Kaggle 10min 400x300 + 50s EEG 400x256 concat ...
cv_kl_all,0.6173
cv_kl_high,0.3228
public_kl,None
private_kl,None
auc_lat,0.9355
auc_pd,0.9234
sens_at_spec90,0.9454


저장 파일 : ['fold0.pt', 'fold0_stage1.pt', 'fold1.pt', 'fold1_stage1.pt', 'fold2.pt', 'fold2_stage1.pt', 'fold3.pt', 'fold3_stage1.pt', 'fold4.pt', 'fold4_stage1.pt', 'metrics.json', 'oof.csv', 'oof_stage1.csv', 'results_row.csv']


## 다음 단계
- **E0 통과 기준** : 셀 3·4가 오류 없이 끝나고 `cv_kl_all`이 숫자로 출력 (값 자체는 의미 없음)
- **E1 실행** : 셀 0의 `EXP = "E1"`, Accelerator **GPU**, Internet **ON** 확인 → Save Version (예상 1~2시간)
- **E2 실행** : `EXP = "E2"` → Save Version (예상 약 0.8시간 : 1단계 0.6h + 2단계 0.2h)
- **E2 완료 후** : 셀 4의 **1단계·2단계 비교표와 H1 판정** 공유 → 지지 시 nb05에서 `EXP="E2"`로 2차 제출
- **E3·E4** : `EXP="E3"`, `SMOKE=True`로 대화형 실행(약 3분) → 오류 없으면 `SMOKE=False`로 바꿔 Save Version (E3 약 0.8시간, E4 약 1.3시간 예상)
  - 메모리 : E3는 eeg50만 (약 5.5GB), E4는 두 배열 모두 (약 9.6GB) 적재
  - 완료 후 공유 : 셀 4 비교표·결과표 한 줄 → H2는 E2·E3·E4 고득표 KL로 판정